In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================================================
# 1. Исходный DataFrame
# =========================================================
# Предполагается, что DataFrame уже загружен и называется df
# Например:
df = pd.read_csv("relevant_data.csv")

# Для безопасности делаем копию
df = df.copy()

In [ ]:
# =========================================================
# 2. Сбор целевой переменной salary
# =========================================================
def build_target_salary(row):
    s_from = row["salary_from"]
    s_to = row["salary_to"]

    if pd.notna(s_from) and pd.notna(s_to):
        return (s_from + s_to) / 2.0
    elif pd.notna(s_from):
        return s_from
    elif pd.notna(s_to):
        return s_to
    else:
        return np.nan

df["target_salary_raw"] = df.apply(build_target_salary, axis=1)

In [ ]:
# =========================================================
# 3. Приведение валют к RUB
# =========================================================
# Подставьте свои курсы, если нужно
currency_rates = {
    "RUR": 1.0,
    "RUB": 1.0,
    "USD": 95.0,
    "EUR": 103.0,
    "KZT": 0.20,
    "BYN": 30.0,
    "BYR": 30.0,
    "UAH": 2.5,
    "UZS": 0.0075,
    "GEL": 35.0,
    "AZN": 56.0,
    "KGS": 1.1,
}

def convert_to_rub(row):
    salary = row["target_salary_raw"]
    currency = row["salary_currency"]

    if pd.isna(salary):
        return np.nan

    if pd.isna(currency):
        currency = "RUR"

    rate = currency_rates.get(currency)
    if rate is None:
        return np.nan

    return salary * rate

df["target_salary"] = df.apply(convert_to_rub, axis=1)

In [ ]:
# =========================================================
# 4. Подготовка признаков
# =========================================================
text_cols = ["name", "description_clean", "key_skills"]

for col in text_cols:
    df[col] = df[col].fillna("").astype(str)

cat_cols = [
    "professional_roles",
    "area_name",
    "experience_name",
    "schedule_name",
    "employment_name",
    "work_format",
    "employer_name",
]

for col in cat_cols:
    df[col] = df[col].fillna("UNKNOWN").astype(str)

df["internship"] = df["internship"].astype(int)

df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
df["pub_month"] = df["published_at"].dt.month.fillna(0).astype(int).astype(str)
df["pub_weekday"] = df["published_at"].dt.weekday.fillna(0).astype(int).astype(str)

df["full_text"] = (
    df["name"] + " [SEP] " +
    df["description_clean"] + " [SEP] " +
    df["key_skills"]
)

extra_cat_cols = cat_cols + ["pub_month", "pub_weekday"]

In [ ]:
# =========================================================
# 5. Датасет только с известной зарплатой
# =========================================================
train_df = df[df["target_salary"].notna()].copy()

# При желании можно убрать выбросы
train_df = train_df[
    (train_df["target_salary"] >= 20000) &
    (train_df["target_salary"] <= 1000000)
].copy()

train_df["target_log"] = np.log1p(train_df["target_salary"])

print("Всего строк в train_df:", len(train_df))

In [ ]:
# =========================================================
# 6. Train / Valid split
# =========================================================
train_part, valid_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

text_col = "full_text"

In [ ]:
# =========================================================
# 7. Подготовка Keras preprocessing слоев
# =========================================================
max_tokens = 20000
seq_len = 250

text_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=seq_len,
    standardize="lower_and_strip_punctuation"
)
text_vectorizer.adapt(train_part[text_col].astype(str).tolist())

lookups = {}
vocab_sizes = {}

for c in extra_cat_cols:
    lookup = tf.keras.layers.StringLookup(output_mode="int", num_oov_indices=1)
    lookup.adapt(train_part[c].astype(str).tolist())
    lookups[c] = lookup
    vocab_sizes[c] = lookup.vocabulary_size()

In [ ]:
# =========================================================
# 8. Функция подготовки входов
# =========================================================
def df_to_inputs(df_):
    out = {
        "text": tf.convert_to_tensor(
            df_[text_col].fillna("").astype(str).tolist(),
            dtype=tf.string
        ),
        "internship": tf.convert_to_tensor(
            df_["internship"].astype(np.float32).values.reshape(-1, 1),
            dtype=tf.float32
        ),
    }

    for c in extra_cat_cols:
        out[c] = tf.convert_to_tensor(
            df_[c].fillna("UNKNOWN").astype(str).tolist(),
            dtype=tf.string
        )

    return out

X_train = df_to_inputs(train_part)
X_valid = df_to_inputs(valid_part)

y_train = tf.convert_to_tensor(
    train_part["target_log"].astype(np.float32).values,
    dtype=tf.float32
)
y_valid = tf.convert_to_tensor(
    valid_part["target_log"].astype(np.float32).values,
    dtype=tf.float32
)

# Проверка типов
for k, v in X_train.items():
    print(k, v.dtype, v.shape)
print("y_train:", y_train.dtype, y_train.shape)

In [ ]:
# =========================================================
# 9. Построение модели
# =========================================================
def make_embedding_for_cat(input_tensor, lookup_layer, vocab_size, emb_dim=16):
    x = lookup_layer(input_tensor)
    x = tf.keras.layers.Embedding(vocab_size + 1, emb_dim)(x)
    x = tf.keras.layers.Flatten()(x)
    return x

# Inputs
text_input = tf.keras.Input(shape=(), dtype=tf.string, name="text")
internship_input = tf.keras.Input(shape=(1,), dtype=tf.float32, name="internship")

cat_inputs = {
    c: tf.keras.Input(shape=(), dtype=tf.string, name=c)
    for c in extra_cat_cols
}

# Text branch
x_text = text_vectorizer(text_input)
x_text = tf.keras.layers.Embedding(max_tokens, 128)(x_text)
x_text = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(64, return_sequences=True)
)(x_text)
x_text = tf.keras.layers.GlobalMaxPooling1D()(x_text)
x_text = tf.keras.layers.Dense(128, activation="relu")(x_text)
x_text = tf.keras.layers.Dropout(0.3)(x_text)

# Categorical branches
cat_embeds = []
for c in extra_cat_cols:
    emb_dim = min(32, max(4, vocab_sizes[c] // 10))
    cat_emb = make_embedding_for_cat(
        cat_inputs[c],
        lookups[c],
        vocab_sizes[c],
        emb_dim=emb_dim
    )
    cat_embeds.append(cat_emb)

# Numeric branch
x_num = tf.keras.layers.Dense(8, activation="relu")(internship_input)

# Combine
x = tf.keras.layers.Concatenate()([x_text, x_num] + cat_embeds)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.35)(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(32, activation="relu")(x)
output = tf.keras.layers.Dense(1, name="salary_log")(x)

model = tf.keras.Model(
    inputs={"text": text_input, "internship": internship_input, **cat_inputs},
    outputs=output
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.Huber(),
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)

model.summary()

In [ ]:
# =========================================================
# 10. Обучение
# =========================================================
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=30,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# =========================================================
# 11. Оценка качества
# =========================================================
pred_valid_log = model.predict(X_valid).reshape(-1)
pred_valid = np.expm1(pred_valid_log)
true_valid = np.expm1(y_valid.numpy())

pred_valid = np.asarray(pred_valid).reshape(-1)
true_valid = np.asarray(true_valid).reshape(-1)

mae = mean_absolute_error(true_valid, pred_valid)

try:
    from sklearn.metrics import root_mean_squared_error
    rmse = root_mean_squared_error(true_valid, pred_valid)
except ImportError:
    rmse = np.sqrt(mean_squared_error(true_valid, pred_valid))

print(f"Validation MAE:  {mae:,.2f} RUB")
print(f"Validation RMSE: {rmse:,.2f} RUB")

In [ ]:
# =========================================================
# 12. Предсказания для всего датасета
# =========================================================
full_df = df.copy()

for col in text_cols:
    full_df[col] = full_df[col].fillna("").astype(str)

for col in cat_cols:
    full_df[col] = full_df[col].fillna("UNKNOWN").astype(str)

full_df["internship"] = full_df["internship"].astype(int)

full_df["published_at"] = pd.to_datetime(full_df["published_at"], errors="coerce")
full_df["pub_month"] = full_df["published_at"].dt.month.fillna(0).astype(int).astype(str)
full_df["pub_weekday"] = full_df["published_at"].dt.weekday.fillna(0).astype(int).astype(str)

full_df["full_text"] = (
    full_df["name"] + " [SEP] " +
    full_df["description_clean"] + " [SEP] " +
    full_df["key_skills"]
)

X_all = df_to_inputs(full_df)

pred_all_log = model.predict(X_all).reshape(-1)
full_df["pred_salary_rub"] = np.expm1(pred_all_log)

# при желании можно ограничить диапазон
full_df["pred_salary_rub"] = full_df["pred_salary_rub"].clip(20000, 1000000)

print(full_df[[
    "name",
    "area_name",
    "experience_name",
    "pred_salary_rub"
]].head())

In [ ]:
# =========================================================
# 13. Сохранение результатов
# =========================================================
result_cols = [
    "vacancy_id",
    "name",
    "area_name",
    "experience_name",
    "employment_name",
    "schedule_name",
    "work_format",
    "employer_name",
    "target_salary",
    "pred_salary_rub"
]

available_result_cols = [c for c in result_cols if c in full_df.columns]
predictions_df = full_df[available_result_cols].copy()

predictions_df.to_csv("vacancy_salary_predictions.csv", index=False)
print("Файл сохранён: vacancy_salary_predictions.csv")